# Phase 1 — TF-IDF Text Representation Experiment

Eksperimen representasi teks berbasis TF-IDF untuk prioritasi kebutuhan perangkat lunak (Functional Requirements dan Non-Functional Requirements). Notebook ini mencakup ekstraksi matriks TF-IDF, seleksi fitur (Chi-Square atau Mutual Information), fusi fitur bisnis (value, effort, risk, stakeholder priority), encoding tipe kebutuhan, pembentukan query group per proyek, pembagian data berbasis proyek, pelatihan model ranking LightGBM LambdaRank, evaluasi metrik (NDCG@5, NDCG@10, MAP, Spearman, Kendall Tau), serta pencatatan artefak dan metrik ke MLflow server.

In [ ]:
import logging
import os
import json
import joblib
import warnings
import itertools
import requests

try:
    from IPython.display import display
except ImportError:
    display = print

import mlflow
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import ndcg_score, average_precision_score
from scipy.stats import spearmanr, kendalltau
import lightgbm as lgb

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
MLFLOW_URI = "https://mlflow.smbgarasibmw.my.id"

server_online = False
try:
    resp = requests.get(MLFLOW_URI, timeout=3)
    if resp.status_code == 200:
        mlflow.set_tracking_uri(MLFLOW_URI)
        mlflow.set_experiment("Phase_1_tfidf_Text_Representation")
        logger.info(f"Successfully connected to MLflow tracking server: {MLFLOW_URI}")
        server_online = True
    else:
        logger.warning(f"MLflow server at {MLFLOW_URI} returned HTTP {resp.status_code}. Using local MLflow store (./mlruns).")
except Exception as e:
    logger.warning(f"MLflow server unreachable ({e}). Using local MLflow store (./mlruns).")

if not server_online:
    mlflow.set_tracking_uri("file:./mlruns")
    mlflow.set_experiment("Phase_1_tfidf_Text_Representation")

## Konfigurasi Parameter Eksperimen

Pengaturan path dataset, direktori keluaran, rasio pembagian data, seed acak, serta parameter ekstraksi dan seleksi fitur TF-IDF.

In [ ]:
DATASET_PATH = "dataset/Dataset_EDA_TFIDF.csv"
OUTPUT_DIR = "outputs/phase1_tfidf"
RANDOM_STATE = 42

# TF-IDF Parameters
TFIDF_MAX_FEATURES = 5000
TFIDF_NGRAM_RANGE = (1, 2)

# Feature Selection Parameters ("chi_square" atau "mutual_information")
FEATURE_SELECTION_METHOD = "chi_square"
K_FEATURES = 500

# Train/Test Split Parameter
TEST_SIZE = 0.2

os.makedirs(OUTPUT_DIR, exist_ok=True)
logger.info(f"Output directory initialized at: {OUTPUT_DIR}")

## STEP 1 — Memuat Dataset dan Validasi Struktur Data

Proses memuat dataset dari CSV, mendeteksi kolom teks kebutuhan secara otomatis, memverifikasi keberadaan kolom bisnis dan ranking yang diperlukan, serta mengurutkan dataset berdasarkan project_id sebagai basis query group.

In [ ]:
logger.info(f"Loading dataset from {DATASET_PATH}...")
df = pd.read_csv(DATASET_PATH)
logger.info(f"Dataset shape: {df.shape}")
logger.info(f"Columns count: {len(df.columns)}")

text_keywords = ["requirement", "text", "description", "sentence", "story", "content"]
text_cols = [c for c in df.columns if any(k in c.lower() for k in text_keywords)]
if text_cols:
    logger.info(f"Detected requirement text column(s): {text_cols}")
    nama_kolom_teks = text_cols[0]
else:
    raise ValueError("No text column found for TF-IDF extraction!")

required_cols = {"id", "project_id", "type", "value", "effort", "risk", "stakeholder_priority", "rank"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

logger.info("All required columns present for TF-IDF ranking pipeline.")

df = df.sort_values(by="project_id").reset_index(drop=True)
logger.info("Dataset sorted by 'project_id' for ranking context.")

display(df.head(3))

## STEP 2 — Validasi Kualitas Data

Pemeriksaan kualitas data yang meliputi identifikasi nilai hilang (missing values), duplikasi baris, duplikasi ID kebutuhan, keabsahan project_id dan nilai rank, serta pengecekan tipe data seluruh kolom.

In [ ]:
validation_report = {}

# Missing values
missing_counts = df.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]
validation_report["missing_values"] = len(missing_cols)
if len(missing_cols) > 0:
    logger.warning(f"Columns with missing values:\n{missing_cols}")
else:
    logger.info("No missing values found.")

# Duplicate rows
dup_rows = df.duplicated().sum()
validation_report["duplicate_rows"] = int(dup_rows)
if dup_rows > 0:
    logger.warning(f"Duplicate rows: {dup_rows}")
else:
    logger.info("No duplicate rows found.")

# Duplicate requirements (by id)
dup_ids = df["id"].duplicated().sum()
validation_report["duplicate_ids"] = int(dup_ids)
if dup_ids > 0:
    logger.warning(f"Duplicate requirement IDs: {dup_ids}")
else:
    logger.info("No duplicate IDs found.")

# Invalid project_id
invalid_pid = df["project_id"].isnull().sum() + (df["project_id"].astype(str).str.strip() == "").sum()
validation_report["invalid_project_ids"] = int(invalid_pid)
if invalid_pid > 0:
    logger.warning(f"Invalid project IDs: {invalid_pid}")
else:
    logger.info("All project IDs are valid.")

# Invalid rank
invalid_rank = (~pd.to_numeric(df["rank"], errors="coerce").notna()).sum()
validation_report["invalid_rank"] = int(invalid_rank)
if invalid_rank > 0:
    logger.warning(f"Invalid rank values: {invalid_rank}")
else:
    logger.info("All rank values are valid.")

validation_report["dtypes"] = {c: str(dt) for c, dt in df.dtypes.items()}

logger.info("=== Validation Report ===")
for k, v in validation_report.items():
    if k != "dtypes":
        logger.info(f"  {k}: {v}")

## STEP 3 — Ekstraksi Matriks Fitur TF-IDF

Mengambil matriks representasi teks TF-IDF yang telah dihitung sebelumnya dari kolom-kolom dataset yang diawali dengan tfidf_.

In [ ]:
tfidf_cols = [c for c in df.columns if c.startswith("tfidf_")]
X_tfidf = df[tfidf_cols].values

logger.info(f"Extracted pre-computed TF-IDF matrix with {X_tfidf.shape[1]} features.")
logger.info(f"TF-IDF matrix shape: {X_tfidf.shape}")

## STEP 4 — Seleksi Fitur Teks (Feature Selection)

Menerapkan SelectKBest dengan metode Chi-Square (chi2) atau Mutual Information (mutual_info_classif) untuk mereduksi dimensi fitur TF-IDF menjadi K fitur teks terbaik yang paling relevan dengan urutan prioritas.

In [ ]:
logger.info(f"Starting feature selection using method: {FEATURE_SELECTION_METHOD} (K={K_FEATURES})...")

y_rank = df["rank"].values

if FEATURE_SELECTION_METHOD == "chi_square":
    selector = SelectKBest(score_func=chi2, k=min(K_FEATURES, X_tfidf.shape[1]))
elif FEATURE_SELECTION_METHOD == "mutual_information":
    selector = SelectKBest(score_func=mutual_info_classif, k=min(K_FEATURES, X_tfidf.shape[1]))
else:
    raise ValueError(f"Metode '{FEATURE_SELECTION_METHOD}' tidak dikenal!")

X_text_selected = selector.fit_transform(X_tfidf, y_rank)

# Ambil nama kolom fitur TF-IDF yang terpilih
selected_indices = selector.get_support(indices=True)
selected_tfidf_cols = [tfidf_cols[i] for i in selected_indices]

logger.info(f"Feature selection completed. Reduced text matrix shape: {X_text_selected.shape}")
logger.info(f"Selected TF-IDF features count: {len(selected_tfidf_cols)}")

## STEP 5 — Fusi Fitur Teks dan Atribut Bisnis (Feature Fusion)

Menggabungkan fitur numerik bisnis (value, effort, risk, stakeholder_priority) dengan fitur teks TF-IDF hasil seleksi secara horizontal untuk membentuk matriks fitur terintegrasi.

In [ ]:
num_features = ["value", "effort", "risk", "stakeholder_priority"]
logger.info(f"Numerical business features to fuse: {num_features}")

X_num = df[num_features].values
X_text_dense = X_text_selected.toarray() if hasattr(X_text_selected, "toarray") else X_text_selected

X_fused = np.hstack((X_num, X_text_dense))
fused_cols = num_features + selected_tfidf_cols

logger.info(f"Fusion completed:")
logger.info(f"  Numerical dimension: {X_num.shape[1]}")
logger.info(f"  Selected text dimension: {X_text_dense.shape[1]}")
logger.info(f"  Fused matrix shape: {X_fused.shape}")

## STEP 6 — Encoding Fitur Kategori dan Pembentukan Matriks Fitur Akhir

Melakukan One-Hot Encoding pada kolom kategori type (FR dan NFR), menggabungkannya ke dalam matriks fitur, dan menyusun DataFrame matriks fitur X dengan nama kolom yang terstruktur.

In [ ]:
logger.info("Performing One-Hot Encoding on 'type' column...")

encoded_type_df = pd.get_dummies(df["type"], prefix="type")
type_cols = list(encoded_type_df.columns)
X_encoded_type = encoded_type_df.values

X_matrix = np.hstack((X_fused, X_encoded_type))
all_feature_cols = fused_cols + type_cols

# Menyusun DataFrame X dengan nama kolom fitur yang valid
X = pd.DataFrame(X_matrix, columns=all_feature_cols)

logger.info(f"Feature encoding completed.")
logger.info(f"  Encoded type dimension: {X_encoded_type.shape[1]}")
logger.info(f"  Final DataFrame X shape: {X.shape}")

## STEP 7 — Pembentukan Query Group dan Label Relevansi

Mengonversi kolom rank menjadi skor relevansi terurut per proyek (0 = prioritas terendah, N-1 = prioritas tertinggi) untuk LambdaRank dan mendefinisikan kelompok proyek (groups) berdasarkan project_id.

In [ ]:
logger.info("Transforming ranks into LambdaRank relevance labels...")

df["label"] = df.groupby("project_id")["rank"].transform(
    lambda x: x.rank(method="dense", ascending=False).astype(int) - 1
)

y = df["label"].values
groups = df["project_id"].values

group_sizes = pd.Series(groups).value_counts()
num_groups = len(group_sizes)

logger.info(f"Number of query groups (projects): {num_groups}")
logger.info(f"Group size stats:\n{group_sizes.describe()}")

single_req_groups = (group_sizes == 1).sum()
if single_req_groups > 0:
    logger.warning(f"Groups with only 1 requirement: {single_req_groups} — these cannot be ranked")

logger.info(f"Label validation -> Range: [{y.min()}, {y.max()}], Unique labels count: {len(np.unique(y))}")

## STEP 8 — Pembagian Data Latih dan Data Uji Berbasis Proyek

Membagi dataset menjadi data latih dan data uji menggunakan GroupShuffleSplit berdasarkan project_id untuk memastikan tidak terjadi kebocoran data antar proyek (no project overlap).

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y[train_idx]
y_test = y[test_idx]
groups_train = groups[train_idx]
groups_test = groups[test_idx]

# Menghitung ukuran setiap grup secara presisi sesuai urutan kemunculan di data latih dan uji
train_group_counts = [len(list(g)) for k, g in itertools.groupby(groups_train)]
test_group_counts = [len(list(g)) for k, g in itertools.groupby(groups_test)]

logger.info(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
logger.info(f"Train groups count: {len(train_group_counts)}, Test groups count: {len(test_group_counts)}")

train_projects = set(groups_train)
test_projects = set(groups_test)
overlap = train_projects & test_projects
if overlap:
    raise ValueError(f"Project overlap detected between train and test sets: {overlap}")
logger.info("No project overlap between train and test sets. Group split is clean.")

## STEP 9 — Pelatihan Model Ranking LightGBM LambdaRank

Melatih model LightGBM Ranker dengan objektivitas lambdarank menggunakan data latih dan grup proyek, serta menampilkan 10 fitur terpenting berdasarkan feature importances.

In [ ]:
max_group_size = max(train_group_counts) if train_group_counts else 31
num_leaves = min(max_group_size, 255)

ranker = lgb.LGBMRanker(
    objective="lambdarank",
    boosting_type="gbdt",
    n_estimators=100,
    num_leaves=num_leaves,
    learning_rate=0.1,
    min_child_samples=10,
    random_state=RANDOM_STATE,
    verbose=-1
)

logger.info("Training LightGBM Ranker...")
ranker.fit(
    X_train, y_train,
    group=train_group_counts,
    eval_set=[(X_test, y_test)],
    eval_group=[test_group_counts],
    eval_metric=["ndcg"],
    callbacks=[lgb.log_evaluation(0)]
)
logger.info("Training complete.")

top_features = pd.Series(ranker.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
logger.info(f"Feature importances (top 10):\n{top_features}")

## STEP 10 — Evaluasi Model dengan Metrik Ranking

Melakukan prediksi pada data uji dan mengukur performa ranking menggunakan metrik NDCG@5, NDCG@10, Mean Average Precision (MAP), korelasi Spearman, dan Kendall Tau.

In [ ]:
y_pred = ranker.predict(X_test)

test_project_ids = groups_test
ndcg5_scores = []
ndcg10_scores = []
map_per_group = []

for pid in np.unique(test_project_ids):
    mask = (test_project_ids == pid)
    y_true_group = y_test[mask]
    y_pred_group = y_pred[mask]
    n = len(y_true_group)

    if n >= 5:
        k5 = min(5, n)
        ndcg5_scores.append(ndcg_score(y_true_group.reshape(1, -1), y_pred_group.reshape(1, -1), k=k5))
    if n >= 10:
        k10 = min(10, n)
        ndcg10_scores.append(ndcg_score(y_true_group.reshape(1, -1), y_pred_group.reshape(1, -1), k=k10))

    if len(np.unique(y_true_group)) > 1:
        threshold = y_true_group.max() * 0.5
        y_bin = (y_true_group >= threshold).astype(int)
        if y_bin.sum() > 0 and y_bin.sum() < len(y_bin):
            map_per_group.append(average_precision_score(y_bin, y_pred_group))

ndcg5 = float(np.mean(ndcg5_scores)) if ndcg5_scores else 0.0
ndcg10 = float(np.mean(ndcg10_scores)) if ndcg10_scores else 0.0
map_score = float(np.mean(map_per_group)) if map_per_group else 0.0

spearman_corr, spearman_p = spearmanr(y_test, y_pred)
kendall_corr, kendall_p = kendalltau(y_test, y_pred)

metrics = {
    "NDCG_at_5": round(float(ndcg5), 6),
    "NDCG_at_10": round(float(ndcg10), 6),
    "MAP": round(float(map_score), 6),
    "Spearman": round(float(spearman_corr), 6),
    "Spearman_pvalue": round(float(spearman_p), 6),
    "KendallTau": round(float(kendall_corr), 6),
    "KendallTau_pvalue": round(float(kendall_p), 6)
}

logger.info("=== Evaluation Metrics ===")
for k, v in metrics.items():
    logger.info(f"  {k}: {v}")

display(pd.DataFrame([metrics]))

## STEP 11 — Penyimpanan Artefak dan Output Eksperimen

Menyimpan model terlatih, dataset hasil pemrosesan, berkas metrik evaluasi JSON, hasil prediksi data uji, serta daftar fitur ke dalam direktori keluaran outputs/phase1_tfidf.

In [ ]:
# 1. Save trained model
model_path = os.path.join(OUTPUT_DIR, "lgbm_ranker.pkl")
joblib.dump(ranker, model_path)
logger.info(f"Model saved: {model_path}")

# 2. Save generated dataset
dataset_out = X.copy()
dataset_out["label"] = y
dataset_out["rank"] = df["rank"].values
dataset_out["project_id"] = groups
dataset_path = os.path.join(OUTPUT_DIR, "tfidf_dataset.csv")
dataset_out.to_csv(dataset_path, index=False)
logger.info(f"Generated dataset saved: {dataset_path} (shape: {dataset_out.shape})")

# 3. Save evaluation metrics
metrics_path = os.path.join(OUTPUT_DIR, "evaluation_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
logger.info(f"Metrics saved: {metrics_path}")

# 4. Save predictions
predictions_df = pd.DataFrame({
    "project_id": groups_test,
    "true_label": y_test,
    "predicted_score": y_pred
})
pred_path = os.path.join(OUTPUT_DIR, "predictions.csv")
predictions_df.to_csv(pred_path, index=False)
logger.info(f"Predictions saved: {pred_path}")

# 5. Save feature list
features_path = os.path.join(OUTPUT_DIR, "feature_list.txt")
with open(features_path, "w") as f:
    f.write("\n".join(X.columns.tolist()))
logger.info(f"Feature list saved: {features_path}")

## STEP 12 — Pencatatan MLflow dan Ringkasan Akhir Eksperimen

Mencatat seluruh parameter eksperimen TF-IDF, metrik performa ranking, dan artefak ke server MLflow tracking https://mlflow.smbgarasibmw.my.id/ serta menampilkan tabel ringkasan eksperimen.

In [ ]:
if mlflow.active_run():
    mlflow.end_run()

try:
    with mlflow.start_run(run_name="TFIDF_LightGBM_Ranker_Experiment") as run:
        # Log parameters
        mlflow.log_param("text_representation", "Pre-Computed TF-IDF")
        mlflow.log_param("feature_selection_method", FEATURE_SELECTION_METHOD)
        mlflow.log_param("selected_k_features", K_FEATURES)
        mlflow.log_param("total_fused_features", X.shape[1])
        mlflow.log_param("train_size", len(X_train))
        mlflow.log_param("test_size", len(X_test))
        mlflow.log_param("num_train_groups", len(train_group_counts))
        mlflow.log_param("num_test_groups", len(test_group_counts))
        mlflow.log_param("random_state", RANDOM_STATE)
        mlflow.log_param("test_size_ratio", TEST_SIZE)
        mlflow.log_param("ranker_objective", "lambdarank")
        mlflow.log_param("ranker_n_estimators", 100)
        mlflow.log_param("ranker_num_leaves", num_leaves)

        # Log metrics
        mlflow.log_metrics(metrics)

        # Log artifacts
        mlflow.log_artifact(model_path, artifact_path="model")
        mlflow.log_artifact(metrics_path, artifact_path="metrics")
        mlflow.log_artifact(features_path, artifact_path="features")
        mlflow.log_artifact(dataset_path, artifact_path="dataset")
        mlflow.log_artifact(pred_path, artifact_path="predictions")

        logger.info(f"MLflow run ID: {run.info.run_id}")
        logger.info("MLflow experiment logged successfully.")
except Exception as e:
    logger.warning(f"MLflow logging error: {e}. Experiment execution & local artifacts completed successfully.")

summary_data = {
    "Metric": ["NDCG@5", "NDCG@10", "MAP", "Spearman rho", "Spearman p-value", "Kendall tau", "Kendall tau p-value"],
    "Value": [
        metrics["NDCG_at_5"],
        metrics["NDCG_at_10"],
        metrics["MAP"],
        metrics["Spearman"],
        metrics["Spearman_pvalue"],
        metrics["KendallTau"],
        metrics["KendallTau_pvalue"]
    ]
}
summary_df = pd.DataFrame(summary_data)

logger.info("=== Experiment Summary ===")
logger.info(f"Text representation: Pre-Computed TF-IDF")
logger.info(f"Feature selection method: {FEATURE_SELECTION_METHOD}")
logger.info(f"Selected TF-IDF features: {K_FEATURES}")
logger.info(f"Total features in X: {X.shape[1]}")
logger.info(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
logger.info(f"Train groups: {len(train_group_counts)}, Test groups: {len(test_group_counts)}")
logger.info(f"\n{summary_df.to_string(index=False)}")

display(summary_df)